<a href="https://colab.research.google.com/github/manuuconrad/elsi-fall-prediction/blob/main/Predicting_Fall_Risk_ELSI_Brazil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

# Definição do caminho do arquivo
# Nota: Este arquivo deve ser solicitado via ELSI-Brasil (https://elsi.cpqrr.fiocruz.br/)
arquivo = 'elsi_data.dta'

# Verificação de segurança para o usuário que baixar seu código
if os.path.exists(arquivo):
    # Carregamento do arquivo .dta (Stata)
    df = pd.read_stata(arquivo)
    print("Dataset carregado com sucesso!")
else:
    print(f"Erro: O arquivo '{arquivo}' não foi encontrado.")
    print("Por favor, garanta que o dataset do ELSI-Brasil esteja na mesma pasta do notebook.")
    # Criamos um dataframe vazio apenas para não quebrar as células seguintes durante o desenvolvimento
    df = pd.DataFrame()


            # dataframe
colunas_n = [c for c in df.columns if c.startswith('n') and c in ['n18', 'n19', 'n21', 'n23']]
df_final = df[['ar10', 'ar8'] + colunas_n].copy()
df_final = df_final.dropna(subset=['n18'])

print("--- DISTRIBUIÇÃO DE QUEDAS NO ELSI ---")
print(df_final['n18'].value_counts())

print("\nIdade média dos entrevistados:", df_final['ar10'].mean())

termos_preditores = ['visão', 'enxerga', 'audição', 'ouve', 'tontura', 'diabetes', 'artrite', 'remédio', 'medicamento', 'sono']
achados_preditores = {}

print("--- BUSCANDO PREDITORES PARA O MODELO ---")
for col, label in labels.items():
    label_min = str(label).lower()
    for termo in termos_preditores:
        if termo in label_min:
            print(f"{col}: {label}")

In [ ]:
import pandas as pd
import numpy as np

# Colunas de interesse
colunas_modelo = ['ar10', 'ar8', 'n18', 'h21', 'n35', 'n56', 't2', 'r4']
df_ml = df[colunas_modelo].copy()
df_ml = df_ml[df_ml['n18'].isin(['Sim', 'NÃ£o'])]

# sim = 1 / nao = 0
mapping = {'Sim': 1, 'NÃ£o': 0, 'Masculino': 1, 'Feminino': 0}
df_ml['target'] = df_ml['n18'].map(mapping)
df_ml['sexo'] = df_ml['ar8'].map(mapping)
df_ml['visao_ruim'] = df_ml['h21'].map(mapping)
df_ml['diabetes'] = df_ml['n35'].map(mapping)
df_ml['artrite'] = df_ml['n56'].map(mapping)
df_ml['sono_ruim'] = df_ml['r4'].map(mapping)

df_ml['medicamentos'] = pd.to_numeric(df_ml['t2'], errors='coerce')

df_final_ml = df_ml[['target', 'ar10', 'sexo', 'visao_ruim', 'diabetes', 'artrite', 'medicamentos', 'sono_ruim']].dropna()

print(f"Dataset pronto com {len(df_final_ml)} idosos.")
print(df_final_ml.head())

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Preditores
X = df_final_ml[['ar10', 'sexo', 'visao_ruim', 'diabetes', 'artrite', 'medicamentos', 'sono_ruim']]
y = df_final_ml['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

# Coeficientes
coeficientes = pd.DataFrame(zip(X.columns, np.exp(modelo.coef_[0])), columns=['Variável', 'Odds Ratio (Risco)'])
print("--- IMPACTO NO RISCO DE QUEDA (Odds Ratio) ---")
print(coeficientes.sort_values(by='Odds Ratio (Risco)', ascending=False))

y_pred = modelo.predict(X_test)
print("\n--- RELATÓRIO DE PRECISÃO DO MODELO ---")
print(classification_report(y_test, y_pred))

# Modelo tunado para lidar com o desequilíbrio
modelo_tunado = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo_tunado.fit(X_train, y_train)


coef_tunado = pd.DataFrame(zip(X.columns, np.exp(modelo_tunado.coef_[0])), columns=['Variável', 'Odds Ratio'])
print("--- NOVO IMPACTO NO RISCO (Modelo Balanceado) ---")
print(coef_tunado.sort_values(by='Odds Ratio', ascending=False))


y_pred_tunado = modelo_tunado.predict(X_test)
print("\n--- NOVO RELATÓRIO DE PRECISÃO ---")
print(classification_report(y_test, y_pred_tunado))


print("\n--- MATRIZ DE CONFUSÃO (Realidade vs Predição) ---")
print(confusion_matrix(y_test, y_pred_tunado))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Preparar os dados (usando os valores que você obteve no modelo balanceado)
df_plot = coef_tunado.sort_values(by='Odds Ratio', ascending=True)

# Tradução para o Inglês (Mais comum para portfólios internacionais)
labels_map = {
    'sono_ruim': 'Poor Sleep Quality',
    'artrite': 'Arthritis Diagnosis',
    'visao_ruim': 'Poor Vision Acuity',
    'diabetes': 'Diabetes Diagnosis',
    'medicamentos': 'Number of Medications',
    'ar10': 'Age (Chronological)',
    'sexo': 'Sex (Male)'
}
df_plot['Variável'] = df_plot['Variável'].map(labels_map)

# 2. Criar o gráfico
plt.figure(figsize=(10, 6))
colors = ['#d9534f' if x > 1.2 else '#5bc0de' for x in df_plot['Odds Ratio']]

ax = sns.barplot(x='Odds Ratio', y='Variável', data=df_plot, palette=colors)

# Adicionar a linha de referência (1.0 = Sem efeito)
plt.axvline(x=1, color='black', linestyle='--', linewidth=1.5)

# Títulos e Labels
plt.title('Predictors of Falls in Older Adults (ELSI-Brazil)\nLogistic Regression - Odds Ratio Model', fontsize=15, pad=20)
plt.xlabel('Odds Ratio (Risk Increase)', fontsize=12)
plt.ylabel('', fontsize=12)

# Adicionar os valores nas barras para facilitar a leitura
for i, v in enumerate(df_plot['Odds Ratio']):
    ax.text(v + 0.02, i, f'{v:.2f}', color='black', va='center', fontweight='bold')

# Estilo acadêmico
sns.despine()
plt.tight_layout()

# Salvar a imagem
plt.savefig('fall_risk_odds_ratio.png', dpi=300)
plt.show()

In [ ]:
def simulador_de_risco(idade, sexo_masc, visao_ruim, diabetes, artrite, medicamentos, sono_ruim):
    paciente = pd.DataFrame([[idade, sexo_masc, visao_ruim, diabetes, artrite, medicamentos, sono_ruim]],
                            columns=['ar10', 'sexo', 'visao_ruim', 'diabetes', 'artrite', 'medicamentos', 'sono_ruim'])

    probabilidade = modelo_tunado.predict_proba(paciente)[0][1]

    # 3. Resultado formatado
    print(f"--- RELATÓRIO DE RISCO DE QUEDA ---")
    print(f"Probabilidade estimada: {probabilidade*100:.1f}%")

    if probabilidade > 0.6:
        print("ALERTA: Alto Risco. Recomenda-se intervenção imediata (Fisioterapia/Revisão de Sono).")
    elif probabilidade > 0.4:
        print("CUIDADO: Risco Moderado. Monitorar fatores ambientais e visão.")
    else:
        print("RISCO BAIXO: Manter acompanhamento de rotina.")

print("Teste do Paciente A:")
simulador_de_risco(idade=69, sexo_masc=0, visao_ruim=0, diabetes=0, artrite=0, medicamentos=5, sono_ruim=1)